# Week 3 — LangChain Basics (Beginner Friendly)

**Goal:** understand the building blocks of a simple LLM app before moving to larger workflows.

**By the end of this notebook, you should be able to:**
- explain what LangChain does
- tell the difference between an LLM and a chat model
- write a simple prompt template
- call a chat model with `invoke`
- understand `batch` and `stream`
- build a tiny Translate → Summarize workflow
- debug the most common API mistakes

## 1) What is LangChain?

LangChain is a framework for building LLM apps step by step.

For beginners, think of it as a toolbox that helps you:
- organize prompts
- call models in a consistent way
- connect multiple steps together
- reuse the same logic in different projects

## 1b) LLM vs Chat Model

**LLM (Language Model):**
- Takes a text prompt, returns text completion
- No concept of "roles" or message structure

**Chat Model:**
- Takes messages with roles (system, user, assistant)
- Returns a message response
- Designed for conversation
- Better for structured workflows

**In this class:** we use chat models because they are clearer and match real-world LLM apps better.

## 2) Must-Know Glossary

- **Prompt:** the instruction you send to the model
- **PromptTemplate:** a reusable prompt with placeholders
- **LLM:** a model that generates text
- **Chat model:** a model that works with messages
- **System message:** the hidden instruction or role
- **User message:** the user's input
- **Chain:** one step feeding into the next step
- **Token:** a chunk of text the model reads or writes
- **Context window:** the maximum amount of text the model can handle at once

## Setup: Install Packages

In [ ]:
# Step 1: Install required packages
!pip install -U langchain langchain-openai python-dotenv

## Setup: Load API Key

In [ ]:
# Step 1: Load the OpenAI API key from .env file
from dotenv import load_dotenv
import os

# Step 2: Load environment variables
load_dotenv()

# Step 3: Check if API key exists
has_key = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY found:", has_key)

# TODO: If False, check your .env file before continuing

## Concept Checks: Building Blocks

In [ ]:
# Step 1: Understand the three main ways to call a model
# Step 2: Print them out
print("1) invoke  -> one prompt, one response")
print("2) batch   -> many prompts, many responses")
print("3) stream  -> response arrives in chunks")

# TODO: For beginners, which one should you start with? Why?

In [ ]:
# Step 1: Create a prompt template with placeholders
template = "Translate the text from {source_lang} to {target_lang}:\n\n{text}"

# Step 2: Fill in the placeholders with .format()
example = template.format(source_lang="English", target_lang="Spanish", text="LangChain is helpful.")

# Step 3: Print the result
print(example)

# TODO: Modify the template to ask for a different language pair

In [ ]:
# Step 1: Messages have roles (system, user, assistant)
# Step 2: System message = behavior instruction
system_message = "You are a helpful tutor."

# Step 3: User message = what the user is asking
user_message = "Explain LangChain in one sentence."

# Step 4: Print both
print("System message:", system_message)
print("User message:", user_message)

# TODO: What would you put in the system message for a customer support bot?

In [ ]:
# Step 1: Create a list of prompts for batch processing
prompts = [
    "Write a one-line definition of LangChain.",
    "Write a one-line definition of a prompt template.",
    "Write a one-line definition of a chat model.",
]

# Step 2: Print the list
print("Batch inputs:")
for item in prompts:
    print("-", item)

# TODO: When would you use batch instead of invoke?

In [ ]:
# Step 1: Understand that structured data can be built from text
import json

# Step 2: Create a JSON string (what the model will return)
raw_text = '{"summary": "likes learning LangChain", "language": "English"}'

# Step 3: Parse it into a Python dictionary
parsed = json.loads(raw_text)

# Step 4: Print the result
print("Parsed JSON:", parsed)
print("Summary:", parsed['summary'])

# TODO: What happens if the JSON is invalid? Try adding a typo and run it again.

## First Real Chat Model Call

In [ ]:
# Step 1: Import the chat model and message classes from LangChain
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

# Step 2: Initialize a chat model (uses OPENAI_API_KEY from .env)
chat = init_chat_model(model='gpt-4o-mini', model_provider='openai')

# Step 3: Create system and user messages
messages = [
    SystemMessage(content='You are a concise assistant that answers clearly.'),
    HumanMessage(content='Summarize the benefits of using LangChain in one short paragraph.')
]

# Step 4: Call the model with invoke() to get one response
resp = chat.invoke(messages)

# Step 5: Extract and print the response content
try:
    print(getattr(resp, 'content', resp))
except Exception:
    print(repr(resp))

# TODO: What did the model return? Does it match your expectation?

## Tokens, Context Window, Temperature

- **Token:** a chunk of text (roughly 4 characters). Models read and write tokens.
- **Context window:** max tokens the model can process at once. gpt-4o-mini: 128k tokens.
- **Temperature:** how creative the model is (0–1). Default: 0.7. Use 0 for deterministic, 1 for creative.

## Prompt Templates with the Chat Model

In [ ]:
# Step 1: Create a translation prompt template
template = '''Translate the following text from {source_lang} to {target_lang}:

{text}

Provide only the translated text.'''

# Step 2: Fill in the template
formatted = template.format(source_lang='English', target_lang='Spanish', text='LangChain helps structure LLM apps.')

# Step 3: Print the formatted prompt
print(formatted)

# Step 4: TODO - Call the chat model with this prompt
# resp = chat.invoke(formatted)
# print(resp.content)

## Few-Shot Prompting + JSON Output Parsing

In [ ]:
# Step 1: Create a few-shot template with examples
import json

few_shot_template = '''You are a JSON responder. Given an input, return a JSON object with keys: 'summary' and 'language'.

Example:
Input: I love programming in Python.
Output: {{"summary": "likes programming", "language": "English"}}

Now respond to:
Input: {text}
Output:'''

# Step 2: Fill in the template
in_text = "Bonjour, j'aime coder en Python."
query = few_shot_template.format(text=in_text)
print('--- Prompt sent to LLM ---')
print(query)

# Step 3: TODO - Call the LLM
# try:
#     resp_obj = chat.invoke(query)
#     resp_text = getattr(resp_obj, 'content', str(resp_obj))
# except Exception as e:
#     resp_text = f'LLM call failed: {e}'

# Step 4: TODO - Print raw response
# print('\n--- Raw LLM response ---')
# print(resp_text)

# Step 5: TODO - Try to parse JSON
# try:
#     parsed = json.loads(resp_text)
#     print('\nParsed JSON:', parsed)
# except Exception as e:
#     print('\nCould not parse JSON:', e)

## Chains & Orchestration: Translate → Summarize

In [ ]:
# Step 1: Define reusable prompt templates
translate_template = 'Translate the following text from {source_lang} to {target_lang}:\n\n{text}\n\nProvide only the translated text.'
summarize_template = 'Summarize the following text in one short paragraph:\n\n{text}\n'

# Step 2: Create a generic LLM calling function with error handling
def call_llm(prompt_text):
    try:
        resp = chat.invoke(prompt_text)
        return getattr(resp, 'content', str(resp))
    except Exception as e:
        return f'LLM error: {e}'

# Step 3: Create specific functions that wrap the templates
def translate(text, source_lang='English', target_lang='Spanish'):
    p = translate_template.format(text=text, source_lang=source_lang, target_lang=target_lang)
    return call_llm(p)

def summarize(text):
    p = summarize_template.format(text=text)
    return call_llm(p)

# Step 4: Test the functions
sample = 'LangChain makes it easier to build reliable LLM-based apps.'

# TODO: Uncomment to test
# translated = translate(sample, source_lang='English', target_lang='Spanish')
# print('--- Translated ---')
# print(translated)

# summary = summarize(translated)
# print('\n--- Summary of translation ---')
# print(summary)

## Debugging Basics

**Four-Step Debugging Process:**
1. Identify where it breaks (API key? Model call? JSON parsing?)
2. Print intermediate values to see what's happening
3. Check assumptions (API key loaded? Model name spelled right?)
4. Google the error message

**Quick Checklist:**
- ✅ Is `.env` in the project root with `OPENAI_API_KEY=sk-...`?
- ✅ Did you run `load_dotenv()` and verify it prints `True`?
- ✅ Is the model name spelled correctly? (gpt-4o-mini, not gpt4o)
- ✅ Print the raw response before parsing
- ✅ Wrap API calls in try-except blocks

## In-Class Exercises

### Exercise 1: Try invoke with different prompts (5 min)

Create a new code cell below and:
- Ask the model to: explain a concept, write code, or tell a joke
- Observe: how does changing the prompt change the response?
- Goal: understand that prompts control the output

In [ ]:
# Step 1: Create a prompt
# Step 2: Call chat.invoke() with the prompt
# Step 3: Print the response

# TODO: Write your code here

### Exercise 2: Use batch (5 min)

In a new cell:
- Create a list of 3–5 prompts
- Use `chat.batch()` to call them all at once
- Compare: is it faster than calling invoke multiple times?
- Goal: understand batch processing

In [ ]:
# Step 1: Create a list of prompts
# Step 2: Call chat.batch() with the list
# Step 3: Print each response

# TODO: Write your code here

### Exercise 3: Try streaming (5 min)

In a new cell:
- Use `chat.stream()` to watch tokens arrive in real time
- Experiment: does streaming give you intermediate results?
- Goal: see the difference between invoke and stream

In [ ]:
# Step 1: Create a prompt
# Step 2: Use a for loop to iterate through chat.stream()
# Step 3: Print each chunk as it arrives

# TODO: Write your code here

### Exercise 4: Structured output (5 min)

In a new cell:
- Write a prompt that asks for JSON output (e.g., name, age, email)
- Parse the response using `json.loads()`
- Handle the case where parsing fails (use try-except)
- Goal: extract structured data from LLM responses

In [ ]:
# Step 1: Create a prompt asking for JSON
# Step 2: Call chat.invoke() with the prompt
# Step 3: Extract the response content
# Step 4: Try to parse it as JSON
# Step 5: Handle errors if parsing fails

# TODO: Write your code here

### Exercise 5: Modify the chain (10 min)

In a new cell:
- Take the Translate → Summarize workflow from earlier
- Add a `tone` parameter (formal, casual, funny)
- Test with different tones
- Goal: understand how to extend existing functions

In [ ]:
# Step 1: Modify the summarize() function to accept a tone parameter
# Step 2: Update the summarize_template to include tone instructions
# Step 3: Test with different tone values (formal, casual, funny)
# Step 4: Print the results

# TODO: Write your code here

## Extension Task: Build a Multi-Step Content Processor

**Goal:** Create a workflow that translates → summarizes → extracts key points (all in one flow).

**Deliverable:** A Python function or notebook that:

1. **Takes a user input:** A block of text in any language
2. **Translates** it to English (if not already)
3. **Summarizes** it in one paragraph
4. **Extracts key points** as a numbered list (structured output)

**Requirements:**
- Use the chat model with system + user messages
- Use prompt templates for all three steps
- Parse the key points as JSON (list of strings)
- Include error handling (API failures, JSON parse errors)
- Test with at least 2 different texts

**Bonus Challenges:**
- Add a `tone` parameter to the summarize step (formal, casual, technical)
- Add a `max_points` parameter to limit key points to N items
- Cache the model responses so you don't re-call for the same input
- Use `batch()` to process multiple texts at once

In [ ]:
# Step 1: Create a translate() function
# Step 2: Create a summarize() function
# Step 3: Create an extract_key_points() function that returns a list
# Step 4: Create a process_content() function that chains them together
# Step 5: Test with at least 2 different texts
# Step 6: Add error handling throughout

# TODO: Write your code here

# Test:
# result = process_content("Your sample text here")
# print(result)